## Архивный ноутбук

В исследовании я использовал YOLO11-seg. В архивных ячейках я сохранил исходные имена весов и параметры; переносимый запуск обучения находится в `train.py`.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

def main():
    # Загрузка модели
    model = YOLO('data/basil/runs/segment/train/weights/best.pt')
    
    # Загрузка изображения
    img_path = 'data/basil/train/images/__00202__IMG_2405.jpg'
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    # Параметры отображения
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 3.0  # Очень крупный текст
    FONT_THICKNESS = 10  # Очень толстый
    TEXT_COLOR = (0, 0, 0)  # Чёрный
    BG_COLOR = (255, 255, 255)  # Белый
    CONTOUR_COLOR = (0, 0, 255)  # Красный (BGR формат)
    CONTOUR_THICKNESS = 8  # Толщина красного контура
    
    # Обработка модели
    results = model(img, imgsz=640)
    
    for i, mask_data in enumerate(results[0].masks.data.cpu().numpy()):
        mask = (mask_data * 255).astype(np.uint8)
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        
        # Расчёт площади
        px_area = cv2.countNonZero(mask)
        cm_area = px_area * (13/w) * (10/h)
        text = f"{cm_area:.1f} cm2"
        
        # Находим контур
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
            
        # Рисуем КРАСНЫЙ контур
        cv2.drawContours(img, contours, -1, CONTOUR_COLOR, CONTOUR_THICKNESS)
        
        # Позиция текста (центр верхней части)
        x = int(np.mean(contours[0][:, 0, 0]))
        y = int(np.min(contours[0][:, 0, 1])) - 50
        
        # Размер текста
        (text_w, text_h), _ = cv2.getTextSize(text, FONT, FONT_SCALE, FONT_THICKNESS)
        
        # Фон для текста
        cv2.rectangle(
            img, 
            (x - text_w//2 - 20, y - text_h - 20), 
            (x + text_w//2 + 20, y + 20), 
            BG_COLOR, -1
        )
        
        # Текст
        cv2.putText(
            img, text, (x - text_w//2, y), 
            FONT, FONT_SCALE, TEXT_COLOR, 
            FONT_THICKNESS, cv2.LINE_AA
        )

    # Сохранение
    output_path = os.path.splitext(img_path)[0] + '_red_contour.jpg'
    cv2.imwrite(output_path, img)
    print(f"Результат с красными контурами сохранён: {output_path}")
    
    # Отображение
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

def main():
    # Загрузка модели
    model = YOLO('data/basil/runs/segment/train/weights/best.pt')
    
    # Загрузка изображения
    img_path = 'data/basil/train/images/__00001__IMG_2306.jpg'
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    # Создаем копию для красивого отображения
    result_img = img.copy()
    
    # Стильные параметры оформления
    CONTOUR_COLORS = [
        (0, 0, 255),    # Красный
        (0, 255, 0),    # Зеленый
        (255, 0, 0),    # Синий
        (255, 255, 0),  # Голубой
        (255, 0, 255),  # Розовый
        (0, 255, 255)   # Желтый
    ]
    
    CONTOUR_THICKNESS = 4
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 1.2
    FONT_THICKNESS = 3
    TEXT_COLOR = (255, 255, 255)  # Белый текст
    BG_COLOR = (40, 40, 40)       # Темно-серый фон
    
    # Обработка модели
    results = model(img, imgsz=640)
    areas = []
    
    if results[0].masks is not None:
        for i, mask_data in enumerate(results[0].masks.data.cpu().numpy()):
            mask = (mask_data * 255).astype(np.uint8)
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            
            # Расчёт площади
            px_area = cv2.countNonZero(mask)
            cm_area = px_area * (13/w) * (10/h)
            areas.append(cm_area)
            
            # Находим контур
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contours:
                continue
            
            # Выбираем цвет для этого листа
            color = CONTOUR_COLORS[i % len(CONTOUR_COLORS)]
            
            # Рисуем красивый контур
            cv2.drawContours(result_img, contours, -1, color, CONTOUR_THICKNESS)
            
            # Рисуем пунктирный внутренний контур
            cv2.drawContours(result_img, contours, -1, color, 1, lineType=cv2.LINE_AA)
            
            # Находим центр масс для текста
            M = cv2.moments(contours[0])
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                
                # Текст с площадью
                text = f"{cm_area:.1f} cm²"
                
                # Размер текста
                (text_w, text_h), baseline = cv2.getTextSize(text, FONT, FONT_SCALE, FONT_THICKNESS)
                
                # Красивый фон для текста с закругленными углами
                padding = 15
                cv2.rectangle(
                    result_img, 
                    (cx - text_w//2 - padding, cy - text_h - padding),
                    (cx + text_w//2 + padding, cy + padding),
                    BG_COLOR, -1, cv2.LINE_AA
                )
                
                # Тень текста
                cv2.putText(result_img, text, 
                          (cx - text_w//2 + 1, cy + 1), 
                          FONT, FONT_SCALE, (0, 0, 0), FONT_THICKNESS, cv2.LINE_AA)
                
                # Основной текст
                cv2.putText(result_img, text, 
                          (cx - text_w//2, cy), 
                          FONT, FONT_SCALE, TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

    # Сохраняем только фотографию с контурами и площадями листьев.
    output_path = os.path.splitext(img_path)[0] + '_area_result.jpg'
    if not cv2.imwrite(output_path, result_img):
        raise OSError(f"Не удалось сохранить: {output_path}")

    plt.figure(figsize=(10, 12))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"Результат сохранён: {output_path}")
    for index, area in enumerate(areas, start=1):
        print(f"Лист {index}: {area:.2f} см²")
    print(f"Общая площадь: {sum(areas):.2f} см²")

if __name__ == "__main__":
    main()
